# Stock Market Analysis Notebook

> **Data note:** this notebook analyzes the project's **synthetic demonstration dataset** (`data/sample_stock_prices.csv`). The companies are fictional and the prices come from a seeded random-walk model, so none of the results describe real companies or real market history.

## 1. Project objective and business questions

**Objective:** practice a complete, explainable price-analysis workflow in Python: load, validate, clean, calculate indicators, compare stocks and summarize findings.

**Business questions**
1. Which stocks gained or lost the most over the whole period?
2. Which stocks were the most volatile, and which were steadier?
3. How deep were the worst peak-to-trough declines (maximum drawdown)?
4. Does trading volume rise on days with large price moves?
5. What do the moving averages and RSI look like for a single stock?

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from IPython.display import display

# Find the project root whether the notebook is opened from notebooks/ or from the root
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT))

from src import market_analysis as ma  # reusable calculations shared with the dashboard

DATA_PATH = ROOT / "data" / "sample_stock_prices.csv"
pd.set_option("display.float_format", "{:,.4f}".format)
pd.set_option("display.width", 140)

## 2. Load the historical dataset

In [ ]:
df = pd.read_csv(DATA_PATH, parse_dates=["Date"])
print(f"Loaded {len(df):,} rows from {DATA_PATH.name}")

## 3. Inspect the data

In [ ]:
display(df.head())

In [ ]:
df.info()

In [ ]:
display(df.describe())

## 4. Validate dates, prices, volume and ticker symbols

`validate_price_data` checks for missing values, non-positive prices, negative volume, impossible High/Low values, duplicate rows and unsorted dates. An empty list means no problems were found.

In [ ]:
issues = ma.validate_price_data(df)
print("Validation issues:", issues if issues else "none found")

per_ticker = df.groupby(["Ticker", "Company"]).agg(
    rows=("Date", "size"),
    first_date=("Date", "min"),
    last_date=("Date", "max"),
)
display(per_ticker)

## 5. Clean and prepare the data

The sample file is already clean, but the cleaning step is kept so the same workflow also works on live data: it removes duplicates and unusable rows, fills a missing `Adjusted_Close` and sorts by ticker and date. Then a wide table of closing prices (one column per ticker) is created for comparisons.

In [ ]:
df = ma.clean_price_data(df)
close_wide = df.pivot(index="Date", columns="Ticker", values="Close")
print(f"{len(df):,} rows after cleaning; wide close table shape: {close_wide.shape}")
display(close_wide.tail())

## 6. Daily returns

Daily return = `Close(t) / Close(t-1) - 1`, calculated separately for each ticker.

In [ ]:
df["Return"] = df.groupby("Ticker")["Close"].transform(ma.daily_returns)

return_stats = df.groupby("Ticker")["Return"].agg(
    mean="mean", std="std", best="max", worst="min"
)
display(return_stats.sort_values("std", ascending=False))

fig = px.box(
    df.dropna(subset=["Return"]).assign(Return_pct=lambda d: d["Return"] * 100),
    x="Ticker", y="Return_pct", title="Distribution of daily returns by ticker (synthetic data)",
    labels={"Return_pct": "Daily return (%)"},
)
fig.show()

## 7. Moving averages

Moving averages smooth past prices. They describe history; they do not predict the future. The 20-day and 50-day averages below only appear once enough observations exist.

In [ ]:
TICKER = "NVLX"  # change this to any ticker in the dataset
one = ma.add_indicators(df[df["Ticker"] == TICKER])

fig = go.Figure()
fig.add_trace(go.Scatter(x=one["Date"], y=one["Close"], name="Close"))
fig.add_trace(go.Scatter(x=one["Date"], y=one["MA_20"], name="20-day MA"))
fig.add_trace(go.Scatter(x=one["Date"], y=one["MA_50"], name="50-day MA"))
fig.update_layout(title=f"{TICKER}: close with 20 and 50-day moving averages (synthetic)", template="plotly_dark")
fig.show()

display(one[["Date", "Close", "MA_20", "MA_50"]].dropna().tail())

## 8. RSI and historical volatility

* **RSI (14-day):** compares the size of recent gains and losses on a 0 to 100 scale. The 30 and 70 levels are conventions, not buy or sell signals.
* **Historical volatility (20-day):** standard deviation of the last 20 daily returns, annualized by multiplying by the square root of 252.

In [ ]:
fig = go.Figure(go.Scatter(x=one["Date"], y=one["RSI"], name="RSI"))
fig.add_hline(y=70, line_dash="dot")
fig.add_hline(y=30, line_dash="dot")
fig.update_layout(title=f"{TICKER}: 14-day RSI (synthetic)", template="plotly_dark", yaxis_range=[0, 100])
fig.show()

latest = []
for t, g in df.groupby("Ticker"):
    ind = ma.add_indicators(g)
    latest.append({
        "Ticker": t,
        "Latest_RSI": ind["RSI"].iloc[-1],
        "Latest_20d_Volatility_%": ind["Volatility"].iloc[-1] * 100,
    })
latest = pd.DataFrame(latest).set_index("Ticker")
display(latest.sort_values("Latest_20d_Volatility_%", ascending=False))

## 9. Compare stock performance

Each closing-price series is divided by its first close and multiplied by 100, so every stock starts at 100 and differences in share price do not matter.

In [ ]:
normalized = ma.normalize_to_base(close_wide)

fig = px.line(normalized, title="Normalized performance, start = 100 (synthetic data)")
fig.update_layout(template="plotly_dark", yaxis_title="Index (start = 100)")
fig.show()

performance = (normalized.iloc[-1] - 100).rename("Total_return_%").sort_values(ascending=False)
display(performance.to_frame())

## 10. Trading volume

Average daily volume by ticker, and the correlation between the size of a day's price move (absolute return) and that day's volume.

In [ ]:
avg_volume = df.groupby("Ticker")["Volume"].mean().sort_values(ascending=False)
fig = px.bar(avg_volume.reset_index(), x="Ticker", y="Volume", title="Average daily volume (synthetic)")
fig.update_layout(template="plotly_dark")
fig.show()

vol_corr = pd.Series(
    {
        t: g["Return"].abs().corr(g["Volume"])
        for t, g in df.dropna(subset=["Return"]).groupby("Ticker")
    },
    name="Corr(|return|, volume)",
)
display(vol_corr.sort_values(ascending=False).to_frame())

## 11. Maximum drawdown

Maximum drawdown is the largest percentage fall from a previous peak close to a later trough close.

In [ ]:
rows = []
for t, g in df.groupby("Ticker"):
    mdd, peak, trough = ma.max_drawdown(g.set_index("Date")["Close"])
    rows.append({"Ticker": t, "Max_drawdown_%": mdd * 100, "Peak_date": peak.date(), "Trough_date": trough.date()})
drawdowns = pd.DataFrame(rows).set_index("Ticker").sort_values("Max_drawdown_%")
display(drawdowns)

## 12. Summary of findings and limitations

The findings below are generated from the data that was just analyzed, so they always match the tables above.

In [ ]:
vol_ann = (
    df.groupby("Ticker")["Return"].std() * np.sqrt(ma.TRADING_DAYS_PER_YEAR) * 100
).sort_values(ascending=False)

best, worst = performance.index[0], performance.index[-1]
print(f"Period analyzed: {df['Date'].min():%Y-%m-%d} to {df['Date'].max():%Y-%m-%d}, {df['Ticker'].nunique()} synthetic tickers")
print(f"- Best performer: {best} ({performance.iloc[0]:+.1f}%). Weakest: {worst} ({performance.iloc[-1]:+.1f}%).")
print(f"- Most volatile: {vol_ann.index[0]} ({vol_ann.iloc[0]:.1f}% annualized). Least volatile: {vol_ann.index[-1]} ({vol_ann.iloc[-1]:.1f}%).")
print(f"- Deepest drawdown: {drawdowns.index[0]} ({drawdowns['Max_drawdown_%'].iloc[0]:.1f}%). Shallowest: {drawdowns.index[-1]} ({drawdowns['Max_drawdown_%'].iloc[-1]:.1f}%).")
print(f"- Highest average volume: {avg_volume.index[0]} ({avg_volume.iloc[0]:,.0f} shares per day).")
print(f"- Median correlation between |daily return| and volume: {vol_corr.median():.2f}")

**Limitations**
* The data is synthetic. The patterns come from the generator's settings (drift, volatility, a simulated sell-off and rebound), not from real events, so they say nothing about actual companies or markets.
* About 18 months of daily data is short for judging long-run behavior.
* Simple returns, simple moving averages and a basic RSI are teaching tools. They are not forecasts or trading signals.
* This notebook is educational and is not investment advice.